# Crypto Volatility Prediction — AE + CNN + LSTM

**모델 목표**: 1분봉 OHLCV 데이터(6시간 lookback)로 향후 30분 실현 변동성 예측

**파이프라인**
```
GCS parquet → DataFrame → on-demand Dataset → AE+CNN+LSTM → 변동성 예측
```

**데이터**: `gs://parkdh0121-ml-data/crypto-vitals/features/`

## 0. 환경 설정

In [ ]:
# GCP 인증 (Colab에서 최초 1회 실행)
from google.colab import auth
auth.authenticate_user()
print('인증 완료')

In [ ]:
# 의존성 설치
!pip install -q google-cloud-storage pyarrow db-dtypes

In [ ]:
# 레포 클론 (최초 1회)
import os
if not os.path.exists('crypto-vitals'):
    !git clone https://github.com/YOUR_USERNAME/crypto-vitals.git
%cd crypto-vitals

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. 데이터 로드 (GCS → DataFrame)

In [ ]:
from google.cloud import storage
import io

GCS_BUCKET = 'parkdh0121-ml-data'
GCS_PREFIX = 'crypto-vitals/features'
PROJECT_ID = 'parkdh0121'

# 학습에 사용할 기간 (Colab RAM 고려: 6개월 이하 권장)
SYMBOL     = 'BTC/USDT'
BLOB_NAME  = f'{GCS_PREFIX}/BTC_USDT/20250604_20260605.parquet'

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(GCS_BUCKET)
blob   = bucket.blob(BLOB_NAME)

buf = io.BytesIO()
blob.download_to_file(buf)
buf.seek(0)
df = pd.read_parquet(buf)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

print(f'Loaded: {df.shape}  [{df.timestamp.min()} → {df.timestamp.max()}]')
df.head(3)

In [ ]:
# RAM 절약: 필요 기간만 슬라이싱 (선택 사항)
# 전체 1년 사용 시 주석 처리
TRAIN_START = '2025-06-04'
TRAIN_END   = '2026-06-05'
df = df[(df.timestamp >= TRAIN_START) & (df.timestamp < TRAIN_END)].reset_index(drop=True)
print(f'Sliced: {df.shape}')

## 2. 데이터 파이프라인

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from ml.data.windows import get_valid_indices
from ml.data.split   import time_split
from ml.data.dataset import make_loaders

LOOKBACK   = 360   # 6시간 입력 윈도우
HORIZON    = 30    # 30분 후 변동성 예측
BATCH_SIZE = 128

indices, ts = get_valid_indices(df, lookback=LOOKBACK, horizon=HORIZON)
splits      = time_split(indices, ts, val_ratio=0.15, test_ratio=0.15)
loaders     = make_loaders(df, splits, lookback=LOOKBACK, horizon=HORIZON,
                           batch_size=BATCH_SIZE, num_workers=2)

# 배치 shape 확인
X_sample, y_sample = next(iter(loaders['train']))
print(f'X batch: {X_sample.shape}   y batch: {y_sample.shape}')
print(f'y stats  min={y_sample.min():.6f}  mean={y_sample.mean():.6f}  max={y_sample.max():.6f}')

## 3. 모델 정의 — AE + CNN + LSTM

In [ ]:
import torch.nn as nn

class DenoisingAE(nn.Module):
    """Denoising Autoencoder: (B, T, F) → (B, T, F)"""
    def __init__(self, n_features: int, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, latent_dim),  nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, n_features),
        )

    def forward(self, x):
        # x: (B, T, F) — apply pointwise along T
        z = self.encoder(x)
        return self.decoder(z), z


class CNNFeatureExtractor(nn.Module):
    """1D-CNN: (B, T, latent) → (B, T', cnn_out)"""
    def __init__(self, in_channels: int, out_channels: int = 64, kernel_size: int = 5):
        super().__init__()
        self.conv = nn.Sequential(
            # (B, C, T) 형식으로 입력
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size // 2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
        )

    def forward(self, x):
        # x: (B, T, C) → permute → (B, C, T)
        return self.conv(x.permute(0, 2, 1)).permute(0, 2, 1)


class VolatilityPredictor(nn.Module):
    """
    AE + CNN + LSTM 하이브리드 변동성 예측 모델.

    1. DenoisingAE : 입력 시퀀스에서 노이즈 제거, 잠재 표현 추출
    2. CNNFeatureExtractor : 로컬 패턴(캔들 형태, 단기 모멘텀) 추출
    3. LSTM : 시간적 의존성 모델링
    4. FC head : 스칼라 변동성 값 출력 (회귀)
    """
    def __init__(
        self,
        n_features:  int = 8,
        ae_latent:   int = 32,
        cnn_out:     int = 64,
        lstm_hidden: int = 128,
        lstm_layers: int = 2,
        dropout:     float = 0.2,
    ):
        super().__init__()
        self.ae  = DenoisingAE(n_features, ae_latent)
        self.cnn = CNNFeatureExtractor(ae_latent, cnn_out)
        self.lstm = nn.LSTM(
            input_size=cnn_out, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 64), nn.ReLU(),
            nn.Linear(64, 1),
            nn.Softplus(),  # 변동성은 항상 양수
        )

    def forward(self, x):
        # x: (B, T, F)
        recon, z = self.ae(x)               # z: (B, T, latent)
        feat     = self.cnn(z)              # feat: (B, T/2, cnn_out)
        out, _   = self.lstm(feat)          # out: (B, T/2, hidden)
        pred     = self.head(out[:, -1, :]) # 마지막 타임스텝 → (B, 1)
        return pred.squeeze(1), recon       # pred: (B,), recon: (B, T, F)


# 모델 초기화 및 파라미터 수 확인
model = VolatilityPredictor().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')
print(model)

## 4. 학습

In [ ]:
# 손실 함수: MSE (예측 변동성) + AE 재구성 손실 (MSE)
# AE 재구성 항은 노이즈 제거 품질을 유지하는 정규화 역할
AE_LOSS_WEIGHT = 0.1

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = pred_loss = ae_loss = 0.0
    with torch.set_grad_enabled(train):
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred, recon = model(X)
            loss_pred = nn.functional.mse_loss(pred, y)
            loss_ae   = nn.functional.mse_loss(recon, X)
            loss      = loss_pred + AE_LOSS_WEIGHT * loss_ae
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item()
            pred_loss  += loss_pred.item()
            ae_loss    += loss_ae.item()
    n = len(loader)
    return total_loss / n, pred_loss / n, ae_loss / n

In [ ]:
EPOCHS    = 30
best_val  = float('inf')
history   = {'train': [], 'val': []}

for epoch in range(1, EPOCHS + 1):
    train_loss, tp, ta = run_epoch(loaders['train'], train=True)
    val_loss,   vp, va = run_epoch(loaders['val'],   train=False)
    scheduler.step(val_loss)

    history['train'].append(train_loss)
    history['val'].append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), 'best_model.pt')

    print(f'Epoch {epoch:3d}/{EPOCHS}  '
          f'train={train_loss:.6f} (pred={tp:.6f} ae={ta:.6f})  '
          f'val={val_loss:.6f} (pred={vp:.6f} ae={va:.6f})')

## 5. 평가

In [ ]:
# 학습 곡선
plt.figure(figsize=(10, 4))
plt.plot(history['train'], label='train')
plt.plot(history['val'],   label='val')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Training Curve'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# 테스트 셋 평가
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
test_loss, tp, _ = run_epoch(loaders['test'], train=False)
print(f'Test MSE (pred): {tp:.8f}')
print(f'Test RMSE:       {tp**0.5:.8f}')

In [ ]:
# 예측값 vs 실제값 시각화
model.eval()
preds, trues = [], []
with torch.no_grad():
    for X, y in loaders['test']:
        pred, _ = model(X.to(DEVICE))
        preds.extend(pred.cpu().numpy())
        trues.extend(y.numpy())

preds = np.array(preds)
trues = np.array(trues)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(trues[:1000], preds[:1000], alpha=0.3, s=5)
lim = max(trues.max(), preds.max())
plt.plot([0, lim], [0, lim], 'r--', lw=1)
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title('Predicted vs Actual (test, first 1000)')

plt.subplot(1, 2, 2)
plt.plot(trues[:500],  label='actual',    alpha=0.7)
plt.plot(preds[:500],  label='predicted', alpha=0.7)
plt.xlabel('Sample'); plt.ylabel('Volatility')
plt.title('Time series (first 500 test samples)')
plt.legend(); plt.grid(True)

plt.tight_layout(); plt.show()

corr = np.corrcoef(trues, preds)[0, 1]
print(f'Pearson r: {corr:.4f}')